In [24]:
import pandas as pd
import numpy as np

In [25]:
cd C:\Users\tr432\Downloads\Figures_GutLungAxis\yes0

C:\Users\tr432\Downloads\Figures_GutLungAxis\yes0


In [26]:
###############################################################################
# 1. Feature table, metadata 읽기
###############################################################################
feat = (pd.read_csv("filtered_table_freq-min-1000.txt", sep="\t", skiprows=1)              # 첫 줄(# Constructed...)만 건너뜀
          .rename(columns={"#OTU ID": "FeatureID"})
          .set_index("FeatureID"))

meta = pd.read_csv("RUN45_Metadata_GutLungAxis.txt", sep="\t")

# feature table 에 실제 존재하는 샘플만 사용
sample_ids = feat.columns.astype(str)
meta_use = meta[meta["#SampleID"].isin(sample_ids)].copy()

# Control / VNAM 만 사용
groups = ["Control", "VNAM"]
group_map = meta_use.set_index("#SampleID")["Group"]
group_map = group_map[group_map.isin(groups)]

In [27]:
feat

,C01-1w,C01-2w,C02-1w,C02-2w,C03-0w,C03-1w,C03-2w,C04-0w,C04-1w,C05-0w,...,V07-2w,V08-0w,V08-1w,V08-2w,V09-0w,V09-1w,V09-2w,V10-0w,V10-1w,V10-2w
FeatureID,,,,,,,,,,,,,,,,,,,,,
ef76567c13e99b2d00fc4403cd44e725,204.0,0.0,0.0,0.0,0.0,0.0,0.0,70.0,91.0,0.0,...,15654.0,130.0,24561.0,10521.0,0.0,20583.0,25530.0,0.0,15562.0,24213.0
e0a561a427aea372fd6aebb418b7351d,0.0,194.0,125.0,0.0,53.0,0.0,53.0,76.0,0.0,213.0,...,14021.0,0.0,21509.0,7323.0,174.0,18011.0,23016.0,123.0,14397.0,21521.0
f140e2b766e53165f8548515a08ae523,0.0,0.0,0.0,203.0,0.0,95.0,0.0,0.0,0.0,0.0,...,11521.0,0.0,17119.0,6613.0,0.0,14647.0,18793.0,0.0,11750.0,17185.0
658f71b9c92766ee310d46b6867fabb4,2819.0,6988.0,3763.0,5431.0,1761.0,2495.0,2961.0,11136.0,7356.0,9323.0,...,0.0,5867.0,96.0,0.0,9255.0,212.0,138.0,5221.0,0.0,188.0
34c88db88b9e0c2c527cbda63accffbb,2562.0,6486.0,3369.0,4794.0,1557.0,2310.0,2545.0,10014.0,6670.0,8433.0,...,0.0,5515.0,0.0,46.0,8506.0,189.0,0.0,4788.0,75.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535a79214f9b84fbb71d0a2d58272cd7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
3c9baa75bc9bf3ab26fae3ff638788d3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
713c20f45567ed600acca28147c959d4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0


In [28]:
meta.head(3)

,#SampleID,subject,Subject_ID,ExpNo,MouseNo,MouseNo2,MouseStrain,Sex,Age,CageNo,...,SacrificeWeek,ExpNo+Group,ExpNo+SamplingWeek,Group+SamplingWeek,ExpNo+Group+SamplingWeek,BM_whole,BM_wbc,Spleen,b.w,Spleen/b.w
0,C01-1w,C01-1w,Exp1_C1-2_1w,Exp1,C1-2,C01,C57BL/6NCrlOri,M,8W,C1,...,Week2,Exp1_Con,Exp1_Week1,Con_Week1,Exp1_Con_Week1,NaN,1960000,64.0,22.8,2.81
1,C01-2w,C01-2w,Exp1_C1-2_2w,Exp1,C1-2,C01,C57BL/6NCrlOri,M,9W,C1,...,Week2,Exp1_Con,Exp1_Week2,Con_Week2,Exp1_Con_Week2,NaN,1960000,64.0,22.8,2.81
2,C02-1w,C02-1w,Exp1_C2-1_1w,Exp1,C2-1,C02,C57BL/6NCrlOri,M,8W,C2,...,Week2,Exp1_Con,Exp1_Week1,Con_Week1,Exp1_Con_Week1,NaN,970000,136.0,23.4,5.81


In [29]:
###############################################################################
# 2. 그룹별 통계 (벤다이어그램과 같은 presence 기준)
###############################################################################
present = {}
n_samples = {}
n_features_union = {}   # Number of Features
n_seqs_total = {}       # Total Frequency
mean_feats = {}
sd_feats = {}
mean_seqs = {}
sd_seqs = {}
min_seqs = {}

for g in groups:
    g_samples = group_map[group_map == g].index
    n_samples[g] = len(g_samples)

    # feature presence: 그룹 내 샘플 중 하나라도 >0 이면 존재
    g_present = (feat[g_samples] > 0).any(axis=1)
    present[g] = g_present
    n_features_union[g] = int(g_present.sum())

    # sample별 read 수 (frequency)
    per_sample_seqs = feat[g_samples].sum(axis=0)
    n_seqs_total[g] = int(per_sample_seqs.sum())
    mean_seqs[g] = per_sample_seqs.mean()
    sd_seqs[g] = per_sample_seqs.std(ddof=1)
    min_seqs[g] = per_sample_seqs.min()

    # sample별 observed feature 수
    per_sample_feats = (feat[g_samples] > 0).sum(axis=0)
    mean_feats[g] = per_sample_feats.mean()
    sd_feats[g] = per_sample_feats.std(ddof=1)

# --- Total (Control + VNAM 전체) ---
all_samples = group_map.index
n_samples["Total"] = len(all_samples)

per_sample_seqs_all = feat[all_samples].sum(axis=0)
n_seqs_total["Total"] = int(per_sample_seqs_all.sum())
mean_seqs["Total"] = per_sample_seqs_all.mean()
sd_seqs["Total"] = per_sample_seqs_all.std(ddof=1)
min_seqs["Total"] = per_sample_seqs_all.min()

per_sample_feats_all = (feat[all_samples] > 0).sum(axis=0)
mean_feats["Total"] = per_sample_feats_all.mean()
sd_feats["Total"] = per_sample_feats_all.std(ddof=1)

present["Total"] = (feat[all_samples] > 0).any(axis=1)
n_features_union["Total"] = int(present["Total"].sum())

In [30]:
###############################################################################
# 3. Venn check (원하면 확인용; Control only / VNAM only / Both)
###############################################################################
ctrl_only = (present["Control"] & ~present["VNAM"]).sum()
vnam_only = (~present["Control"] &  present["VNAM"]).sum()
both      = (present["Control"] &  present["VNAM"]).sum()
print("Venn check  Control only / VNAM only / Both:",
      ctrl_only, vnam_only, both)

Venn check  Control only / VNAM only / Both: 3303 827 1259


In [31]:
###############################################################################
# 4. 요약 테이블 (스크린샷 형태: Total / Control / VNAM)
###############################################################################
def fmt_int(x):
    return f"{int(round(x)):,}"

def fmt_mean_sd(m, s):
    return f"{int(round(m)):,}±{int(round(s)):,}"

row_labels = [
    "Number of Samples",
    "Number of Features",
    "Mean number of Features ± std dev.",
    "Total Frequency",
    "Mean Frequency ± std dev.",
    "Minimum frequency"
]

cols = ["Total", "Control", "VNAM"]

vals = {
    g: [
        fmt_int(n_samples[g]),
        fmt_int(n_features_union[g]),
        fmt_mean_sd(mean_feats[g], sd_feats[g]),
        fmt_int(n_seqs_total[g]),
        fmt_mean_sd(mean_seqs[g], sd_seqs[g]),
        fmt_int(min_seqs[g]),
    ]
    for g in cols
}

summary_table = pd.DataFrame({
    "": row_labels,
    "Total":   vals["Total"],
    "Control": vals["Control"],
    "VNAM":    vals["VNAM"],
})

summary_table.to_csv("sequence_table_Group.csv", index=False, encoding="cp949")

summary_table

,,Total,Control,VNAM
0,Number of Samples,52,26,26
1,Number of Features,"5,389","4,562","2,086"
2,Mean number of Features ± std dev.,290±229,441±156,140±188
3,Total Frequency,"4,100,450","2,196,675","1,903,775"
4,Mean Frequency ± std dev.,"78,855±27,286","84,488±32,148","73,222±20,479"
5,Minimum frequency,"20,731","20,731","24,576"


In [32]:
import pandas as pd
import numpy as np

In [33]:
cd C:\Users\tr432\Downloads\Figures_GutLungAxis\yes0

C:\Users\tr432\Downloads\Figures_GutLungAxis\yes0


In [34]:
###############################################################################
# 1. Feature table, metadata 읽기
###############################################################################
feat = (pd.read_csv("filtered_table_freq-min-1000.txt", sep="\t", skiprows=1)              # 첫 줄(# Constructed...)만 건너뜀
          .rename(columns={"#OTU ID": "FeatureID"})
          .set_index("FeatureID"))

meta = pd.read_csv("RUN45_Metadata_GutLungAxis.txt", sep="\t")

# feature table 에 실제 존재하는 샘플만 사용
sample_ids = feat.columns.astype(str)
meta_use = meta[meta["#SampleID"].isin(sample_ids)].copy()

# Control / VNAM 만 사용
groups = ["Week0", "Week1", "Week2"]
group_map = meta_use.set_index("#SampleID")["SamplingWeek"]
group_map = group_map[group_map.isin(groups)]

In [35]:
feat

,C01-1w,C01-2w,C02-1w,C02-2w,C03-0w,C03-1w,C03-2w,C04-0w,C04-1w,C05-0w,...,V07-2w,V08-0w,V08-1w,V08-2w,V09-0w,V09-1w,V09-2w,V10-0w,V10-1w,V10-2w
FeatureID,,,,,,,,,,,,,,,,,,,,,
ef76567c13e99b2d00fc4403cd44e725,204.0,0.0,0.0,0.0,0.0,0.0,0.0,70.0,91.0,0.0,...,15654.0,130.0,24561.0,10521.0,0.0,20583.0,25530.0,0.0,15562.0,24213.0
e0a561a427aea372fd6aebb418b7351d,0.0,194.0,125.0,0.0,53.0,0.0,53.0,76.0,0.0,213.0,...,14021.0,0.0,21509.0,7323.0,174.0,18011.0,23016.0,123.0,14397.0,21521.0
f140e2b766e53165f8548515a08ae523,0.0,0.0,0.0,203.0,0.0,95.0,0.0,0.0,0.0,0.0,...,11521.0,0.0,17119.0,6613.0,0.0,14647.0,18793.0,0.0,11750.0,17185.0
658f71b9c92766ee310d46b6867fabb4,2819.0,6988.0,3763.0,5431.0,1761.0,2495.0,2961.0,11136.0,7356.0,9323.0,...,0.0,5867.0,96.0,0.0,9255.0,212.0,138.0,5221.0,0.0,188.0
34c88db88b9e0c2c527cbda63accffbb,2562.0,6486.0,3369.0,4794.0,1557.0,2310.0,2545.0,10014.0,6670.0,8433.0,...,0.0,5515.0,0.0,46.0,8506.0,189.0,0.0,4788.0,75.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535a79214f9b84fbb71d0a2d58272cd7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
3c9baa75bc9bf3ab26fae3ff638788d3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
713c20f45567ed600acca28147c959d4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0


In [36]:
meta.head(3)

,#SampleID,subject,Subject_ID,ExpNo,MouseNo,MouseNo2,MouseStrain,Sex,Age,CageNo,...,SacrificeWeek,ExpNo+Group,ExpNo+SamplingWeek,Group+SamplingWeek,ExpNo+Group+SamplingWeek,BM_whole,BM_wbc,Spleen,b.w,Spleen/b.w
0,C01-1w,C01-1w,Exp1_C1-2_1w,Exp1,C1-2,C01,C57BL/6NCrlOri,M,8W,C1,...,Week2,Exp1_Con,Exp1_Week1,Con_Week1,Exp1_Con_Week1,NaN,1960000,64.0,22.8,2.81
1,C01-2w,C01-2w,Exp1_C1-2_2w,Exp1,C1-2,C01,C57BL/6NCrlOri,M,9W,C1,...,Week2,Exp1_Con,Exp1_Week2,Con_Week2,Exp1_Con_Week2,NaN,1960000,64.0,22.8,2.81
2,C02-1w,C02-1w,Exp1_C2-1_1w,Exp1,C2-1,C02,C57BL/6NCrlOri,M,8W,C2,...,Week2,Exp1_Con,Exp1_Week1,Con_Week1,Exp1_Con_Week1,NaN,970000,136.0,23.4,5.81


In [37]:
###############################################################################
# 2. 그룹별 통계 (벤다이어그램과 같은 presence 기준)
###############################################################################
present = {}
n_samples = {}
n_features_union = {}   # Number of Features
n_seqs_total = {}       # Total Frequency
mean_feats = {}
sd_feats = {}
mean_seqs = {}
sd_seqs = {}
min_seqs = {}

for g in groups:
    g_samples = group_map[group_map == g].index
    n_samples[g] = len(g_samples)

    # feature presence: 그룹 내 샘플 중 하나라도 >0 이면 존재
    g_present = (feat[g_samples] > 0).any(axis=1)
    present[g] = g_present
    n_features_union[g] = int(g_present.sum())

    # sample별 read 수 (frequency)
    per_sample_seqs = feat[g_samples].sum(axis=0)
    n_seqs_total[g] = int(per_sample_seqs.sum())
    mean_seqs[g] = per_sample_seqs.mean()
    sd_seqs[g] = per_sample_seqs.std(ddof=1)
    min_seqs[g] = per_sample_seqs.min()

    # sample별 observed feature 수
    per_sample_feats = (feat[g_samples] > 0).sum(axis=0)
    mean_feats[g] = per_sample_feats.mean()
    sd_feats[g] = per_sample_feats.std(ddof=1)

# --- Total (Control + VNAM 전체) ---
all_samples = group_map.index
n_samples["Total"] = len(all_samples)

per_sample_seqs_all = feat[all_samples].sum(axis=0)
n_seqs_total["Total"] = int(per_sample_seqs_all.sum())
mean_seqs["Total"] = per_sample_seqs_all.mean()
sd_seqs["Total"] = per_sample_seqs_all.std(ddof=1)
min_seqs["Total"] = per_sample_seqs_all.min()

per_sample_feats_all = (feat[all_samples] > 0).sum(axis=0)
mean_feats["Total"] = per_sample_feats_all.mean()
sd_feats["Total"] = per_sample_feats_all.std(ddof=1)

present["Total"] = (feat[all_samples] > 0).any(axis=1)
n_features_union["Total"] = int(present["Total"].sum())

In [ ]:
###############################################################################
# 3. Venn check (원하면 확인용; Control only / VNAM only / Both)
###############################################################################
w0 = (present["Week0"] & ~present["Week1"] & ~present["Week2"]).sum()
w1 = (present["Week1"] & ~present["Week0"] & ~present["Week2"]).sum()
w2 = (present["Week2"] & ~present["Week0"] & ~present["Week1"]).sum()

print("Venn check  w0 / w1 / w2:",
      w0, w1, w2)

Venn check  w0 / w1 / w2 / w0+w1 / w1+w2 / w2+w0 / w0+w1+w2: 1624 1083 888 2707 1971 2512 3595


In [39]:
###############################################################################
# 4. 요약 테이블 (스크린샷 형태: Total / Control / VNAM)
###############################################################################
def fmt_int(x):
    return f"{int(round(x)):,}"

def fmt_mean_sd(m, s):
    return f"{int(round(m)):,}±{int(round(s)):,}"

row_labels = [
    "Number of Samples",
    "Number of Features",
    "Mean number of Features ± std dev.",
    "Total Frequency",
    "Mean Frequency ± std dev.",
    "Minimum frequency"
]

cols = ["Total", "Week0", "Week1", "Week2"]

vals = {
    g: [
        fmt_int(n_samples[g]),
        fmt_int(n_features_union[g]),
        fmt_mean_sd(mean_feats[g], sd_feats[g]),
        fmt_int(n_seqs_total[g]),
        fmt_mean_sd(mean_seqs[g], sd_seqs[g]),
        fmt_int(min_seqs[g]),
    ]
    for g in cols
}

summary_table = pd.DataFrame({
    "": row_labels,
    "Total": vals["Total"],
    "Week0": vals["Week0"],
    "Week1": vals["Week1"],
    "Week2": vals["Week2"]
})

summary_table.to_csv("sequence_table_SamplingWeek.csv", index=False, encoding="cp949")

summary_table

,,Total,Week0,Week1,Week2
0,Number of Samples,52,15,21,16
1,Number of Features,"5,389","3,201","2,621","2,272"
2,Mean number of Features ± std dev.,290±229,432±138,235±214,231±266
3,Total Frequency,"4,100,450","1,372,695","1,414,194","1,313,561"
4,Mean Frequency ± std dev.,"78,855±27,286","91,513±29,638","67,343±23,355","82,098±25,047"
5,Minimum frequency,"20,731","20,731","24,576","38,725"


In [40]:
import pandas as pd
import numpy as np

In [41]:
cd C:\Users\tr432\Downloads\Figures_GutLungAxis\yes0

C:\Users\tr432\Downloads\Figures_GutLungAxis\yes0


In [42]:
###############################################################################
# 1. Feature table, metadata 읽기
###############################################################################
feat = (pd.read_csv("filtered_table_freq-min-1000.txt", sep="\t", skiprows=1)              # 첫 줄(# Constructed...)만 건너뜀
          .rename(columns={"#OTU ID": "FeatureID"})
          .set_index("FeatureID"))

meta = pd.read_csv("RUN45_Metadata_GutLungAxis.txt", sep="\t")

# feature table 에 실제 존재하는 샘플만 사용
sample_ids = feat.columns.astype(str)
meta_use = meta[meta["#SampleID"].isin(sample_ids)].copy()

# Control / VNAM 만 사용
groups = ["Con_Week0", "Con_Week1", "Con_Week2", "VNAM_Week0", "VNAM_Week1", "VNAM_Week2"]
group_map = meta_use.set_index("#SampleID")["Group+SamplingWeek"]
group_map = group_map[group_map.isin(groups)]

In [43]:
feat

,C01-1w,C01-2w,C02-1w,C02-2w,C03-0w,C03-1w,C03-2w,C04-0w,C04-1w,C05-0w,...,V07-2w,V08-0w,V08-1w,V08-2w,V09-0w,V09-1w,V09-2w,V10-0w,V10-1w,V10-2w
FeatureID,,,,,,,,,,,,,,,,,,,,,
ef76567c13e99b2d00fc4403cd44e725,204.0,0.0,0.0,0.0,0.0,0.0,0.0,70.0,91.0,0.0,...,15654.0,130.0,24561.0,10521.0,0.0,20583.0,25530.0,0.0,15562.0,24213.0
e0a561a427aea372fd6aebb418b7351d,0.0,194.0,125.0,0.0,53.0,0.0,53.0,76.0,0.0,213.0,...,14021.0,0.0,21509.0,7323.0,174.0,18011.0,23016.0,123.0,14397.0,21521.0
f140e2b766e53165f8548515a08ae523,0.0,0.0,0.0,203.0,0.0,95.0,0.0,0.0,0.0,0.0,...,11521.0,0.0,17119.0,6613.0,0.0,14647.0,18793.0,0.0,11750.0,17185.0
658f71b9c92766ee310d46b6867fabb4,2819.0,6988.0,3763.0,5431.0,1761.0,2495.0,2961.0,11136.0,7356.0,9323.0,...,0.0,5867.0,96.0,0.0,9255.0,212.0,138.0,5221.0,0.0,188.0
34c88db88b9e0c2c527cbda63accffbb,2562.0,6486.0,3369.0,4794.0,1557.0,2310.0,2545.0,10014.0,6670.0,8433.0,...,0.0,5515.0,0.0,46.0,8506.0,189.0,0.0,4788.0,75.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535a79214f9b84fbb71d0a2d58272cd7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
3c9baa75bc9bf3ab26fae3ff638788d3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
713c20f45567ed600acca28147c959d4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0


In [44]:
meta.head(3)

,#SampleID,subject,Subject_ID,ExpNo,MouseNo,MouseNo2,MouseStrain,Sex,Age,CageNo,...,SacrificeWeek,ExpNo+Group,ExpNo+SamplingWeek,Group+SamplingWeek,ExpNo+Group+SamplingWeek,BM_whole,BM_wbc,Spleen,b.w,Spleen/b.w
0,C01-1w,C01-1w,Exp1_C1-2_1w,Exp1,C1-2,C01,C57BL/6NCrlOri,M,8W,C1,...,Week2,Exp1_Con,Exp1_Week1,Con_Week1,Exp1_Con_Week1,NaN,1960000,64.0,22.8,2.81
1,C01-2w,C01-2w,Exp1_C1-2_2w,Exp1,C1-2,C01,C57BL/6NCrlOri,M,9W,C1,...,Week2,Exp1_Con,Exp1_Week2,Con_Week2,Exp1_Con_Week2,NaN,1960000,64.0,22.8,2.81
2,C02-1w,C02-1w,Exp1_C2-1_1w,Exp1,C2-1,C02,C57BL/6NCrlOri,M,8W,C2,...,Week2,Exp1_Con,Exp1_Week1,Con_Week1,Exp1_Con_Week1,NaN,970000,136.0,23.4,5.81


In [45]:
###############################################################################
# 2. 그룹별 통계 (벤다이어그램과 같은 presence 기준)
###############################################################################
present = {}
n_samples = {}
n_features_union = {}   # Number of Features
n_seqs_total = {}       # Total Frequency
mean_feats = {}
sd_feats = {}
mean_seqs = {}
sd_seqs = {}
min_seqs = {}

for g in groups:
    g_samples = group_map[group_map == g].index
    n_samples[g] = len(g_samples)

    # feature presence: 그룹 내 샘플 중 하나라도 >0 이면 존재
    g_present = (feat[g_samples] > 0).any(axis=1)
    present[g] = g_present
    n_features_union[g] = int(g_present.sum())

    # sample별 read 수 (frequency)
    per_sample_seqs = feat[g_samples].sum(axis=0)
    n_seqs_total[g] = int(per_sample_seqs.sum())
    mean_seqs[g] = per_sample_seqs.mean()
    sd_seqs[g] = per_sample_seqs.std(ddof=1)
    min_seqs[g] = per_sample_seqs.min()

    # sample별 observed feature 수
    per_sample_feats = (feat[g_samples] > 0).sum(axis=0)
    mean_feats[g] = per_sample_feats.mean()
    sd_feats[g] = per_sample_feats.std(ddof=1)

# --- Total (Control + VNAM 전체) ---
all_samples = group_map.index
n_samples["Total"] = len(all_samples)

per_sample_seqs_all = feat[all_samples].sum(axis=0)
n_seqs_total["Total"] = int(per_sample_seqs_all.sum())
mean_seqs["Total"] = per_sample_seqs_all.mean()
sd_seqs["Total"] = per_sample_seqs_all.std(ddof=1)
min_seqs["Total"] = per_sample_seqs_all.min()

per_sample_feats_all = (feat[all_samples] > 0).sum(axis=0)
mean_feats["Total"] = per_sample_feats_all.mean()
sd_feats["Total"] = per_sample_feats_all.std(ddof=1)

present["Total"] = (feat[all_samples] > 0).any(axis=1)
n_features_union["Total"] = int(present["Total"].sum())

In [46]:
###############################################################################
# 4. 요약 테이블 (스크린샷 형태: Total / Control / VNAM)
###############################################################################
def fmt_int(x):
    return f"{int(round(x)):,}"

def fmt_mean_sd(m, s):
    return f"{int(round(m)):,}±{int(round(s)):,}"

row_labels = [
    "Number of Samples",
    "Number of Features",
    "Mean number of Features ± std dev.",
    "Total Frequency",
    "Mean Frequency ± std dev.",
    "Minimum frequency"
]

cols = ["Total", "Con_Week0", "Con_Week1", "Con_Week2", "VNAM_Week0", "VNAM_Week1", "VNAM_Week2"]

vals = {
    g: [
        fmt_int(n_samples[g]),
        fmt_int(n_features_union[g]),
        fmt_mean_sd(mean_feats[g], sd_feats[g]),
        fmt_int(n_seqs_total[g]),
        fmt_mean_sd(mean_seqs[g], sd_seqs[g]),
        fmt_int(min_seqs[g]),
    ]
    for g in cols
}

summary_table = pd.DataFrame({
    "": row_labels,
    "Total": vals["Total"],
    "Con_Week0": vals["Con_Week0"],
    "Con_Week1": vals["Con_Week1"],
    "Con_Week2": vals["Con_Week2"],
    "VNAM_Week0": vals["VNAM_Week0"],
    "VNAM_Week1": vals["VNAM_Week1"],
    "VNAM_Week2": vals["VNAM_Week2"]
})

summary_table.to_csv("sequence_table_Group+SamplingWeek.csv", index=False, encoding="cp949")

summary_table

,,Total,Con_Week0,Con_Week1,Con_Week2,VNAM_Week0,VNAM_Week1,VNAM_Week2
0,Number of Samples,52,9,11,6,6,10,10
1,Number of Features,"5,389","2,272","2,438","2,140","1,769",334,219
2,Mean number of Features ± std dev.,290±229,409±164,404±156,556±95,466±91,48±29,36±8
3,Total Frequency,"4,100,450","818,935","761,074","616,666","553,760","653,120","696,895"
4,Mean Frequency ± std dev.,"78,855±27,286","90,993±37,876","69,189±24,263","102,778±26,500","92,293±12,769","65,312±23,436","69,690±14,116"
5,Minimum frequency,"20,731","20,731","33,664","59,458","79,470","24,576","38,725"
